In [3]:
import pandas as pd
from src.data.raw_data import download_data
from src.data.process_data import deduplicate_categories
from src.utils.config import VALIDATED_DATA_PATH

In [ ]:
data = download_data()
data.head(5)

In [ ]:
# Validate nulls
null_before_clean = data.isnull().sum()
display('-Validation nulls count:-', null_before_clean)

# Handle nulls
data.dropna(subset=['genres', 'categories'], inplace=True)
data.fillna('Unknown', inplace=True)
data.reset_index(drop=True, inplace=True)

null_after_clean = data.isnull().sum()
display('-Handled nulls count:-', null_after_clean)

In [ ]:
# 'name' column Validation
data['name'] = data['name'].str.strip()

In [ ]:
# 'release_year' and  'release_date' columns validation
data_years = data['release_year'].unique()
study_years = [2021, 2022, 2023, 2024, 2025]
if sorted(data_years) != study_years:
    data = data[data['release_year'].isin(study_years)]
print(f"Unique years in 'release_year': {data_years}")

data['release_date'] = pd.to_datetime(data['release_date'], errors='coerce')
print(f"Number of missing release dates: {data['release_date'].isna().sum()}")
data.dropna(subset=['release_date'], inplace=True)
data.reset_index(drop=True, inplace=True)

In [ ]:
# 'genres' and 'categories' columns validation
data = deduplicate_categories(data, 'genres')
data = deduplicate_categories(data, 'categories')
data.reset_index(drop=True, inplace=True)
data.shape

In [ ]:
# 'price' and 'recommendations' columns validation
negative_prices = data['price'] < 0
negative_recommendations = data['recommendations'] < 0
data = data[~(negative_prices | negative_recommendations)]
print(f"Removed {negative_prices.sum()} negative prices and {negative_recommendations.sum()} negative recommendations")

In [ ]:
# 'developer' and 'publisher' columns validation
data['developer'] = data['developer'].str.strip()
data['publisher'] = data['publisher'].str.strip()

In [ ]:
# Save validated data
print(f"Data final shape: {data.shape}")
data.to_parquet(f'{VALIDATED_DATA_PATH}', index=False)